In [1]:
import ir_datasets
from nltk.stem import WordNetLemmatizer
import nltk
from helper import create_tokenized_word_list

dataset = ir_datasets.load("wikir/en1k/training")
tokenized = create_tokenized_word_list(dataset)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
nltk.download("punkt_tab")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/tahas44/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
lemmatized_tokenized_docs = []
lematizer = WordNetLemmatizer()

for query_list in tokenized:
    lemmatized_words = [lematizer.lemmatize(word) for word in query_list]
    lemmatized_tokenized_docs.append(lemmatized_words)

print("Docs lemmatized")

Docs lemmatized


In [4]:
from helper import create_tokenized_word_list_for_query
query_tokenized = create_tokenized_word_list_for_query(dataset)
print("Queries tokenized!")

Queries tokenized!


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
lemmatized_tokenized_queries = []

for query_list in query_tokenized:
    lemmatized_words = [lematizer.lemmatize(words) for words in query_list]
    lemmatized_tokenized_queries.append(lemmatized_words)

print("Queries lemmatized")

Queries lemmatized


In [6]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi(lemmatized_tokenized_docs)

In [7]:
scores_of_all_queries = []

for query in lemmatized_tokenized_queries:
    scores_of_all_queries.append(bm25.get_scores(query))

In [8]:
from collections import defaultdict
from helper import Scoredoc

doc_dict = defaultdict(str)

for i, doc in enumerate(dataset.docs_iter()):
    doc_dict[i] = doc.doc_id

doc_dict = dict(doc_dict)

qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

score_doc_dict = dict(score_doc_dict)

print("Necessary dicts created!")

Necessary dicts created!


In [ ]:
query_ids = [query.query_id for query in dataset.queries_iter()]
query_ids

['123839',
 '188629',
 '13898',
 '316959',
 '515031',
 '123783',
 '4332',
 '9591',
 '114982',
 '563603',
 '104206',
 '1580851',
 '23678',
 '37480',
 '109454',
 '544250',
 '12519',
 '115015',
 '84287',
 '24998',
 '11925',
 '643',
 '113075',
 '1250387',
 '72012',
 '167865',
 '1425120',
 '24883',
 '27200',
 '136856',
 '6031',
 '1313191',
 '187115',
 '84110',
 '208234',
 '6431',
 '10799',
 '1417144',
 '32512',
 '677637',
 '97937',
 '113765',
 '12900',
 '1294909',
 '73745',
 '11692',
 '22173',
 '76662',
 '164301',
 '23344',
 '9367',
 '592217',
 '98546',
 '98175',
 '15189',
 '1256',
 '1815937',
 '1249076',
 '80274',
 '38489',
 '11311',
 '4754',
 '257711',
 '141668',
 '100895',
 '32945',
 '112321',
 '133725',
 '424464',
 '523456',
 '1327515',
 '12101',
 '9739',
 '189869',
 '143444',
 '78940',
 '12527',
 '327199',
 '7707',
 '97455',
 '1262433',
 '488875',
 '74026',
 '358108',
 '95179',
 '7862',
 '151797',
 '92980',
 '100876',
 '238228',
 '107755',
 '32539',
 '12953',
 '24843',
 '119528',
 '307

In [10]:
from collections import defaultdict

query_id_vs_text = defaultdict(str)

for query in dataset.queries_iter():
    query_id_vs_text[query.query_id] = query.text

In [ ]:
import numpy as np
query_id_and_its_doc = defaultdict(list)

for i in range(len(query_ids)):
    found_doc_list = []
    similarity = scores_of_all_queries[i]

    index_of_found_docs = np.where(similarity > 0)[0]

    for index in list(index_of_found_docs):
        found_doc_list.append((similarity[index], doc_dict[index]))

    if len(found_doc_list) <= 10:
        for mytuple in sorted(found_doc_list, reverse=True):
            query_id_and_its_doc[query_ids[i]].append(mytuple[1])
    else:
        for mytuple in sorted(found_doc_list, reverse=True):
            query_id_and_its_doc[query_ids[i]].append(mytuple[1])

            if len(query_id_and_its_doc[query_ids[i]]) == 10:
                break

In [12]:
import gc
del tokenized
del lematizer
del query_tokenized
gc.collect()

0

In [13]:
del dataset
del lemmatized_words
gc.collect()

0

In [14]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")

doc_id_vs_text = defaultdict(str)

for doc in dataset.docs_iter():
    doc_id_vs_text[doc.doc_id] = doc.text

In [15]:
del dataset
gc.collect()

0

In [16]:
query_doc_pairs = defaultdict(list)

for query_id in query_ids:
    for doc_id in query_id_and_its_doc[query_id]:
        query_doc_pairs[query_id].append((query_id_vs_text[query_id], doc_id_vs_text[doc_id]))

In [17]:
from sentence_transformers import CrossEncoder
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

/run/media/tahas44/Yeni Birim/Technarts/Intern/NLP/InformationRetrieval/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2522.95it/s]


In [18]:
dict_keys_set = set(query_doc_pairs.keys())

In [19]:
query_score_dict = defaultdict(list)

for query_id in query_ids:
    if query_id in dict_keys_set:
        query_score_dict[query_id] = list(model.predict(query_doc_pairs[query_id]))

In [20]:
query_score_dict["123839"]

[np.float32(6.4627924),
 np.float32(6.419842),
 np.float32(4.9677753),
 np.float32(3.0357656),
 np.float32(5.5189133),
 np.float32(2.3268332),
 np.float32(4.384878),
 np.float32(0.7513137),
 np.float32(-3.6361036),
 np.float32(-4.7235107)]

In [21]:
del query_doc_pairs
gc.collect()

133

In [22]:
most_related_3_dict = defaultdict(list)
most_related_3_dict_with_scores = defaultdict(list)

for query_id in query_ids:
    if query_id in dict_keys_set:
        related_10 = []

        for i, score in enumerate(query_score_dict[query_id]):
            related_10.append((score, i))

        for mytuple in sorted(related_10, reverse=True):
            most_related_3_dict[query_id].append(query_id_and_its_doc[query_id][mytuple[1]])
            most_related_3_dict_with_scores[query_id].append((query_id_and_its_doc[query_id][mytuple[1]], mytuple[0]))

            if len(most_related_3_dict[query_id]) == 3:
                break

In [23]:
most_related_3_dict

defaultdict(list,
            {'123839': ['806300', '123839', '806075'],
             '188629': ['188629', '2185399', '1990307'],
             '13898': ['13898', '2213954', '1698181'],
             '316959': ['316959', '1326263', '930698'],
             '515031': ['515031', '888195', '832086'],
             '123783': ['1210005', '1209576', '123783'],
             '4332': ['4332', '1442076', '989116'],
             '9591': ['143444', '1898638', '2416624'],
             '114982': ['2305139', '2305216', '1341103'],
             '563603': ['12846', '563603', '349302'],
             '104206': ['1281151', '1972988', '104018'],
             '1580851': ['1549040', '2218736', '2187550'],
             '23678': ['23678', '936323', '342272'],
             '37480': ['77763', '305953', '1027733'],
             '109454': ['109454', '2069260', '2100641'],
             '544250': ['544250', '2095411', '630265'],
             '12519': ['1091625', '1739665', '1005711'],
             '115015': ['115015', '

In [24]:
lemmatized_docs = []

for lemma_token_doc in lemmatized_tokenized_docs:
    lemmatized_docs.append(" ".join(lemma_token_doc))

lemmatized_docs

['used landing craft world war ii used today private boat training facility 6 71 inline six cylinder diesel engine 71 refers displacement cubic inch cylinder firing order engine 1 5 3 6 2 4 engine compression ratio 18 7 1 4 250 inch bore 5 00 inch stroke engine weighs 54 inch long 29 inch wide 41 inch tall 2 100 revolution per minute engine capable producing 230 horse power 172 kilowatt v type version 71 series developed 1957 6 71 two stroke engine engine naturally aspirate air provided via root type blower however 6 71t model turbocharger supercharger utilized fuel provided unit injector one per cylinder amount fuel injected engine controlled engine governor engine cooling via liquid water jacket boat cool external water pumped engine',
 'rejecting offer cambridge university moved london 1954 working mayfair advertising agency moonlighting hat check girl night club le club contemporain working royal college art met painter frank bowling still student married 1960 one son kitchen one w

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf_vect = TfidfVectorizer()
tf_idf_matrix = tf_idf_vect.fit_transform(lemmatized_docs)
print("TF-IDF matrix ready!")

TF-IDF matrix ready!


In [26]:
type(tf_idf_vect)

sklearn.feature_extraction.text.TfidfVectorizer

In [27]:
tf_idf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 32486544 stored elements and shape (369721, 772603)>

In [28]:
type(tf_idf_matrix)

scipy.sparse._csr.csr_matrix

In [29]:
tf_idf_vect.get_feature_names_out()

array(['00', '000', '0000', ..., 'zzt', 'zzx', 'zzz'],
      shape=(772603,), dtype=object)

In [30]:
import numpy as np
import scipy
import sklearn.feature_extraction.text

def get_top_word(tf_idf_matrix: scipy.sparse._csr.csr_matrix, 
                 tf_idf_vect: sklearn.feature_extraction.text.TfidfVectorizer,
                 doc_index: int):

    row = tf_idf_matrix[doc_index]

    if row.nnz == 0:
        return None

    max_index = np.argmax(row.data)
    top_feature_index = row.indices[max_index]
    feature_names = tf_idf_vect.get_feature_names_out()
    top_term = feature_names[top_feature_index]

    return top_term

In [31]:
for key, value in doc_dict.items():
    print(key, value)
    break

0 1781133


In [32]:
doc_id_vs_index = defaultdict(int)

for key, value in doc_dict.items():
    doc_id_vs_index[value] = key

doc_id_vs_index = dict(doc_id_vs_index)

In [33]:
doc_id_vs_index

{'1781133': 0,
 '2426736': 1,
 '2224122': 2,
 '219642': 3,
 '1728654': 4,
 '1889917': 5,
 '1518473': 6,
 '1459418': 7,
 '1102956': 8,
 '931408': 9,
 '1043905': 10,
 '488327': 11,
 '1069841': 12,
 '1601336': 13,
 '1415191': 14,
 '1531137': 15,
 '912052': 16,
 '935016': 17,
 '1671729': 18,
 '1946935': 19,
 '752649': 20,
 '1634473': 21,
 '867034': 22,
 '1064413': 23,
 '1612321': 24,
 '2222141': 25,
 '1784663': 26,
 '2285161': 27,
 '1445565': 28,
 '1469713': 29,
 '1466972': 30,
 '963723': 31,
 '897432': 32,
 '1218994': 33,
 '839556': 34,
 '2170795': 35,
 '1024270': 36,
 '1336432': 37,
 '2087053': 38,
 '421312': 39,
 '547088': 40,
 '636479': 41,
 '2166683': 42,
 '833090': 43,
 '2363592': 44,
 '2289234': 45,
 '1370298': 46,
 '1481866': 47,
 '1763284': 48,
 '1876607': 49,
 '729433': 50,
 '352078': 51,
 '2354924': 52,
 '2416742': 53,
 '1283366': 54,
 '1602376': 55,
 '1301443': 56,
 '1962423': 57,
 '2436983': 58,
 '79923': 59,
 '490168': 60,
 '502902': 61,
 '1738343': 62,
 '52687': 63,
 '503290

In [34]:
doc_id_vs_index["806300"]

170443

In [35]:
get_top_word(tf_idf_matrix, tf_idf_vect, 170443)

'yanni'

In [36]:
doc_id_vs_text["806300"]

'it is a compilation of tracks from his five previous studio albums released between 1980 and 1989 plus three new compositions yanni was encouraged to release the album by his then partner actress linda evans reflections of passion became yanni s fastest selling and most successful album of his career upon release it reached no 1 on the billboard top new age albums chart and no 29 on the billboard 200 yanni supported the album with a nationwide concert tour in 1990 that featured his band and an orchestra in 1995 it was certified double platinum for selling 2 million copies in the us in august 1989 yanni released his fifth studio album niki nana the album marked his stylistic development from solo keyboard music towards rock with the addition of additional vocalists musicians and choir around the same time of its release yanni s newfound relationship with american actress linda evans who had become a fan of his music received press attention not long into their relationship evans pitche

In [37]:
counter = 0

for word in doc_id_vs_text["806300"].split():
    if word == "yanni":
        counter += 1

counter

6

In [38]:
lemmatized_tokenized_queries

[['yanni'],
 ['k', 'pop'],
 ['venice', 'film', 'festival'],
 ['downtown', 'brooklyn'],
 ['pennsylvania', 'house', 'representative'],
 ['northern', 'premier', 'league'],
 ['first', 'national', 'picture'],
 ['poland'],
 ['cyperaceae'],
 ['trinidad', 'tobago'],
 ['normandy', 'landing'],
 ['perry', 'index'],
 ['arm', 'architecture'],
 ['mi5'],
 ['patriarch', 'antioch'],
 ['nashik', 'district'],
 ['semiotics'],
 ['santa', 'cruz', 'de', 'tenerife'],
 ['gastropoda'],
 ['zaragoza'],
 ['south', 'carolina'],
 ['augustine', 'hippo'],
 ['canton', 'geneva'],
 ['leo', 'tolstoy'],
 ['riemannian', 'manifold'],
 ['tuzla'],
 ['subaru'],
 ['tomsk'],
 ['united', 'state', 'department', 'agriculture'],
 ['north', 'borneo'],
 ['isaac', 'asimov'],
 ['hilbert', 'space'],
 ['chechen'],
 ['arunachal', 'pradesh'],
 ['central', 'otago'],
 ['ira', 'gershwin'],
 ['rock', 'roll'],
 ['fiba'],
 ['east', 'coast', 'united', 'state'],
 ['wikipedia'],
 ['pernambuco'],
 ['west', 'coast', 'eagle'],
 ['twin', 'peak'],
 ['dutc

In [39]:
del most_related_3_dict_with_scores
del query_score_dict
del score_doc_dict
gc.collect()

20

In [40]:
query_id_vs_index = defaultdict(int)

for i, query_id in enumerate(query_ids):
    query_id_vs_index[query_id] = i

In [41]:
query_id_vs_index = dict(query_id_vs_index)

In [42]:
len(most_related_3_dict.keys())

1442

In [43]:
del lemmatized_docs
gc.collect()

0

In [44]:
queries_expanded = []

for query_id in query_ids:
    if query_id in dict_keys_set:
        doc_list = most_related_3_dict[query_id]
        query_list = lemmatized_tokenized_queries[query_id_vs_index[query_id]]
        query_list *= 2

        for doc_id in doc_list:
            added_word = get_top_word(tf_idf_matrix, tf_idf_vect, doc_id_vs_index[doc_id])

            if added_word is not None:
                query_list.append(added_word)

        queries_expanded.append(query_list)
    else:
        queries_expanded.append([])

In [45]:
new_scores = []

for query in queries_expanded:
    new_scores.append(bm25.get_scores(query))

In [46]:
import numpy as np
query_id_and_its_doc = defaultdict(list)

for i in range(len(query_ids)):
    found_doc_list = []
    similarity = new_scores[i]

    index_of_found_docs = np.where(similarity > 0)[0]

    for index in list(index_of_found_docs):
        found_doc_list.append((similarity[index], doc_dict[index]))

    if len(found_doc_list) <= 10:
        for mytuple in found_doc_list:
            query_id_and_its_doc[query_ids[i]].append(mytuple[1])
    else:
        for mytuple in sorted(found_doc_list, reverse=True):
            query_id_and_its_doc[query_ids[i]].append(mytuple[1])

            if len(query_id_and_its_doc[query_ids[i]]) == 10:
                break

In [47]:
query_doc_pairs = defaultdict(list)

for query_id in query_ids:
    for doc_id in query_id_and_its_doc[query_id]:
        query_doc_pairs[query_id].append((query_id_vs_text[query_id], doc_id_vs_text[doc_id]))

In [48]:
query_score_dict = defaultdict(list)

for query_id in query_ids:
    if query_id in dict_keys_set:
        query_score_dict[query_id] = list(model.predict(query_doc_pairs[query_id]))

In [49]:
most_related_5_dict = defaultdict(list)
most_related_5_dict_with_scores = defaultdict(list)

for query_id in query_ids:
    if query_id in dict_keys_set:
        related_10 = []

        for i, score in enumerate(query_score_dict[query_id]):
            related_10.append((score, i))

        for mytuple in sorted(related_10, reverse=True):
            most_related_5_dict[query_id].append(query_id_and_its_doc[query_id][mytuple[1]])
            most_related_5_dict_with_scores[query_id].append((query_id_and_its_doc[query_id][mytuple[1]], mytuple[0]))

            if len(most_related_5_dict[query_id]) == 5:
                break

In [50]:
most_related_10_dict = defaultdict(list)
most_related_10_dict_with_scores = defaultdict(list)

for query_id in query_ids:
    if query_id in dict_keys_set:
        related_10 = []

        for i, score in enumerate(query_score_dict[query_id]):
            related_10.append((score, i))

        for mytuple in sorted(related_10, reverse=True):
            most_related_10_dict[query_id].append(query_id_and_its_doc[query_id][mytuple[1]])
            most_related_10_dict_with_scores[query_id].append((query_id_and_its_doc[query_id][mytuple[1]], mytuple[0]))

            if len(most_related_10_dict[query_id]) == 10:
                break

In [51]:
def recall(found_docs: list, test_list: list) -> float:
    counter = 0
    test_set = set(test_list)

    for doc_id in found_docs:
        if doc_id in test_set:
            counter += 1

    return (counter / len(test_list)) * 100

def precision(found_docs: list, test_list: list) -> float:
    counter = 0
    test_set = set(test_list)

    for doc_id in found_docs:
        if doc_id in test_set:
            counter += 1

    return (counter / len(found_docs)) * 100

def precision_AP(found_docs: list, test_list: list) -> float:
    counter = 0

    test_set = set(test_list)

    for doc in found_docs:
        if doc in test_set:
            counter += 1

    return counter / len(found_docs)


def AP(found_docs: list, test_list: list) -> float:
    total = 0

    for i in range(1, 10 + 1):
        if i < len(found_docs):
            precision_k = precision_AP(found_docs[:i], test_list)
            total += precision_k * (found_docs[i - 1] in set(test_list))

    return total / len(test_list)

In [52]:
recall_5_list, precision_5_list, AP_5_list = [], [], []

for query_id in query_ids:
    if query_id in dict_keys_set:
        recall_5_value = recall(most_related_5_dict[query_id], qrels_dict[query_id])
        precision_5_value = precision(most_related_5_dict[query_id], qrels_dict[query_id])
        AP_5_value = AP(most_related_5_dict[query_id], qrels_dict[query_id])

        recall_5_list.append(recall_5_value)
        precision_5_list.append(precision_5_value)
        AP_5_list.append(AP_5_value)
    else:
        recall_5_list.append(0)
        precision_5_list.append(0)
        AP_5_list.append(0)

In [53]:
recall_10_list, precision_10_list, AP_10_list = [], [], []

for query_id in query_ids:
    if query_id in dict_keys_set:
        recall_10_value = recall(most_related_10_dict[query_id], qrels_dict[query_id])
        precision_10_value = precision(most_related_10_dict[query_id], qrels_dict[query_id])
        AP_10_value = AP(most_related_10_dict[query_id], qrels_dict[query_id])

        recall_10_list.append(recall_10_value)
        precision_10_list.append(precision_10_value)
        AP_10_list.append(AP_10_value)
    else:
        recall_10_list.append(0)
        precision_10_list.append(0)
        AP_10_list.append(0)

In [54]:
import pandas as pd

df = pd.DataFrame({
    "Query_ID": query_ids,
    "recall_10": recall_10_list,
    "precision_10": precision_10_list,
    "AP_10": AP_10_list
})

df

,Query_ID,recall_10,precision_10,AP_10
0,123839,100.000000,60.0,0.915079
1,188629,50.000000,30.0,0.388889
2,13898,33.333333,20.0,0.333333
3,316959,22.222222,20.0,0.222222
4,515031,21.428571,30.0,0.126786
...,...,...,...,...
1439,896124,12.500000,10.0,0.062500
1440,12319,4.545455,10.0,0.045455
1441,4421,0.000000,0.0,0.000000
1442,296526,0.000000,0.0,0.000000


In [56]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

score_doc_dict = dict(score_doc_dict)

In [57]:
from sklearn.metrics import ndcg_score

doc_id_score_dict = defaultdict(float)
ndcg_10_list = []

for idx in range(len(df)):
    query_id = df.loc[idx, "Query_ID"]
    if query_id in dict_keys_set:
        scoredoc_object_list = score_doc_dict[query_id]

        for scoreddoc_object in scoredoc_object_list:
            doc_id_score_dict[scoreddoc_object.doc_id] = scoreddoc_object.score

        model_score_tuple_list = most_related_10_dict_with_scores[query_id]

        y_score, y_true = [], []

        for mytuple in model_score_tuple_list:
            doc_id = mytuple[0]
            score = mytuple[1]

            y_score.append(score)
            y_true.append(doc_id_score_dict[doc_id])

        if len(y_score) == 0:
            ndcg_10_list.append(0.0)
            continue

        if len(y_score) == 1:
            y_true.append(0.0)
            y_score.append(0.0)

        if len(y_true) == len(y_score):
            ndcg_10_list.append(ndcg_score(np.asarray([y_true]), np.asarray([y_score]), k=10))
        else:
            print("There is a problem with query", query_id)
    else:
        ndcg_10_list.append(0)

In [58]:
df["NDCG_10"] = ndcg_10_list
df

,Query_ID,recall_10,precision_10,AP_10,NDCG_10
0,123839,100.000000,60.0,0.915079,0.995386
1,188629,50.000000,30.0,0.388889,0.998432
2,13898,33.333333,20.0,0.333333,0.000000
3,316959,22.222222,20.0,0.222222,0.986344
4,515031,21.428571,30.0,0.126786,0.766278
...,...,...,...,...,...
1439,896124,12.500000,10.0,0.062500,0.990022
1440,12319,4.545455,10.0,0.045455,0.911338
1441,4421,0.000000,0.0,0.000000,0.989373
1442,296526,0.000000,0.0,0.000000,0.651697


In [59]:
df["f_score_10"] = 2 * df["recall_10"] * df["precision_10"] / (df["recall_10"] + df["precision_10"])
df

,Query_ID,recall_10,precision_10,AP_10,NDCG_10,f_score_10
0,123839,100.000000,60.0,0.915079,0.995386,75.000000
1,188629,50.000000,30.0,0.388889,0.998432,37.500000
2,13898,33.333333,20.0,0.333333,0.000000,25.000000
3,316959,22.222222,20.0,0.222222,0.986344,21.052632
4,515031,21.428571,30.0,0.126786,0.766278,25.000000
...,...,...,...,...,...,...
1439,896124,12.500000,10.0,0.062500,0.990022,11.111111
1440,12319,4.545455,10.0,0.045455,0.911338,6.250000
1441,4421,0.000000,0.0,0.000000,0.989373,NaN
1442,296526,0.000000,0.0,0.000000,0.651697,NaN


In [60]:
recall_5_list, precision_5_list, AP_5_list = [], [], []

for query_id in query_ids:
    if query_id in dict_keys_set:
        recall_5_value = recall(most_related_5_dict[query_id], qrels_dict[query_id])
        precision_5_value = precision(most_related_5_dict[query_id], qrels_dict[query_id])
        AP_5_value = AP(most_related_5_dict[query_id], qrels_dict[query_id])

        recall_5_list.append(recall_5_value)
        precision_5_list.append(precision_5_value)
        AP_5_list.append(AP_5_value)
    else:
        recall_5_list.append(0)
        precision_5_list.append(0)
        AP_5_list.append(0)

In [61]:
df["recall_5"] = recall_5_list
df["precision_5"] = precision_5_list
df["AP_5"] = AP_5_list

In [62]:
df["f_score_5"] = 2 * df["recall_5"] * df["precision_5"] / (df["recall_5"] + df["precision_5"])

In [63]:
from sklearn.metrics import ndcg_score

doc_id_score_dict = defaultdict(float)
ndcg_5_list = []

for idx in range(len(df)):
    query_id = df.loc[idx, "Query_ID"]
    if query_id in dict_keys_set:
        scoredoc_object_list = score_doc_dict[query_id]

        for scoreddoc_object in scoredoc_object_list:
            doc_id_score_dict[scoreddoc_object.doc_id] = scoreddoc_object.score

        model_score_tuple_list = most_related_5_dict_with_scores[query_id]

        y_score, y_true = [], []

        for mytuple in model_score_tuple_list:
            doc_id = mytuple[0]
            score = mytuple[1]

            y_score.append(score)
            y_true.append(doc_id_score_dict[doc_id])

        if len(y_score) == 0:
            ndcg_5_list.append(0.0)
            continue

        if len(y_score) == 1:
            y_true.append(0.0)
            y_score.append(0.0)

        if len(y_true) == len(y_score):
            ndcg_5_list.append(ndcg_score(np.asarray([y_true]), np.asarray([y_score]), k=5))
        else:
            print("There is a problem with query", query_id)
    else:
        ndcg_5_list.append(0)

In [64]:
df["NDCG_5"] = ndcg_5_list

In [65]:
mydict = {
    "Method": "BM25PseudoRelevanceFeedback",
    "recall_5_mean": df["recall_5"].mean(),
    "recall_5_std": df["recall_5"].std(),
    "recall_5_max": df["recall_5"].max(),
    "recall_5_min": df["recall_5"].min(),
    "recall_10_mean": df["recall_10"].mean(),
    "recall_10_std": df["recall_10"].std(),
    "recall_10_max": df["recall_10"].max(),
    "recall_10_min": df["recall_10"].min(),
    "precision_5_mean": df["precision_5"].mean(),
    "precision_5_std": df["precision_5"].std(),
    "precision_5_max": df["precision_5"].max(),
    "precision_5_min": df["precision_5"].min(),
    "precision_10_mean": df["precision_10"].mean(),
    "precision_10_std": df["precision_10"].std(),
    "precision_10_max": df["precision_10"].max(),
    "precision_10_min": df["precision_10"].min(),
    "f_score_5_mean": df["f_score_5"].mean(),
    "f_score_5_std": df["f_score_5"].std(),
    "f_score_5_max": df["f_score_5"].max(),
    "f_score_5_min": df["f_score_5"].min(),
    "f_score_10_mean": df["f_score_10"].mean(),
    "f_score_10_std": df["f_score_10"].std(),
    "f_score_10_max": df["f_score_10"].max(),
    "f_score_10_min": df["f_score_10"].min(),
    "MAP_5": df["AP_5"].mean(),
    "MAP_10": df["AP_10"].mean(),
    "NDCG_5_mean": df["NDCG_5"].mean(),
    "NDCG_5_std": df["NDCG_5"].std(),
    "NDCG_5_max": df["NDCG_5"].max(),
    "NDCG_5_min": df["NDCG_5"].min(),
    "NDCG_10_mean": df["NDCG_10"].mean(),
    "NDCG_10_std": df["NDCG_10"].std(),
    "NDCG_10_max": df["NDCG_10"].max(),
    "NDCG_10_min": df["NDCG_10"].min()
}

In [66]:
df_parquet = pd.DataFrame(mydict, index=[0])
df_parquet

,Method,recall_5_mean,recall_5_std,recall_5_max,recall_5_min,recall_10_mean,recall_10_std,recall_10_max,recall_10_min,precision_5_mean,...,MAP_5,MAP_10,NDCG_5_mean,NDCG_5_std,NDCG_5_max,NDCG_5_min,NDCG_10_mean,NDCG_10_std,NDCG_10_max,NDCG_10_min
0,BM25PseudoRelevanceFeedback,16.024472,14.739417,83.333333,0.0,21.604635,20.202779,100.0,0.0,33.067867,...,0.123887,0.157476,0.79951,0.331981,1.0,0.0,0.808502,0.291541,1.0,0.0


In [67]:
df_parquet.to_parquet("BM25PseudoRelevanceFeedback.parquet")